# Survie du groupe
Idée : les passagers d'un même ticket (famille, groupe de voyage) ont souvent partagé le même sort.
36 % des passagers du test ont un ticket déjà vu dans le train : on connaît le sort de leur groupe.

Deux règles, appliquées par-dessus un modèle de base :
- une **femme** dont toutes les femmes et tous les garçons connus du groupe sont morts → **morte** ;
- un **garçon** (Master) dont toutes les femmes et tous les garçons connus du groupe ont survécu → **survivant**.

On ne regarde que femmes et garçons du groupe : les hommes adultes mouraient presque tous, leur sort ne dit rien du groupe.

**CV** : stratifiée, pas groupée. Avec `group_cv`, aucun passager de validation n'a de groupe connu dans le train (0 %),
les règles ne s'appliqueraient jamais. La CV stratifiée (34 % de groupes connus) ressemble au test (36 %).
Pas de fuite : à chaque fold, les règles ne lisent que la survie des passagers du train de ce fold.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from titanic.features import GroupSurvival, group_rules, load_data, make_column_transformer, save_submission

warnings.filterwarnings("ignore")
X, y, test = load_data()

## Le signal
Survie réelle selon le sort des **autres** passagers du même ticket (le passager lui-même est exclu).

In [2]:
gs = GroupSurvival().fit_transform(X, y).ravel()
d = pd.DataFrame({"Sex": X["Sex"], "y": y})
d["groupe"] = pd.cut(gs, [-.01, .001, .499, .501, .999, 1],
                     labels=["tous morts", "plutôt morts", "inconnu", "plutôt vivants", "tous vivants"])
d.pivot_table(index="groupe", columns="Sex", values="y", aggfunc=["mean", "size"], observed=True).round(2)

mean         size     
Sex            female  male female male
groupe                                 
tous morts       0.38  0.07     72   70
plutôt morts     1.00  0.25      3    4
inconnu          0.78  0.16    153  418
plutôt vivants   0.78  0.64      9   11
tous vivants     1.00  0.41     77   74

## Cross-validation
Quatre modèles sur les mêmes 25 découpages : genre seul, GBC, et chacun avec les règles.

In [3]:
pipe = Pipeline([
    ("prep", make_column_transformer("title")),
    ("scaler", StandardScaler()),
    ("imputer", IterativeImputer(estimator=Ridge(), max_iter=1000, random_state=0, tol=1e-3)),
    ("model", GradientBoostingClassifier(random_state=0)),
])

res = {"genre seul": [], "genre + règles": [], "GBC": [], "GBC + règles": []}
for a, b in RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0).split(X, y):
    Xa, ya, Xb, yb = X.iloc[a], y.iloc[a], X.iloc[b], y.iloc[b].values
    genre = (Xb["Sex"] == "female").astype(int).values
    gbc = clone(pipe).fit(Xa, ya).predict(Xb)
    for nom, p in [("genre seul", genre), ("genre + règles", group_rules(Xa, ya, Xb, genre)),
                   ("GBC", gbc), ("GBC + règles", group_rules(Xa, ya, Xb, gbc))]:
        res[nom].append((p == yb).mean())

for nom, s in res.items():
    print(f"{nom:15s} CV : {np.mean(s):.4f} ± {np.std(s):.4f}")

genre seul      CV : 0.7867 ± 0.0331
genre + règles  CV : 0.8368 ± 0.0254
GBC             CV : 0.8310 ± 0.0229
GBC + règles    CV : 0.8364 ± 0.0198


## Soumissions
Règles calculées sur les 891 lignes du train, appliquées au test.

In [4]:
genre_test = (test["Sex"] == "female").astype(int).values
p1 = group_rules(X, y, test, genre_test)
gbc_test = clone(pipe).fit(X, y).predict(test)
p2 = group_rules(X, y, test, gbc_test)

for nom, base, p in [("genre_regles", genre_test, p1), ("gbc_regles", gbc_test, p2)]:
    path = save_submission(test, p, nom)
    print(f"{path.name} : {(p != base).sum()} prédictions changées par les règles, {p.mean():.1%} de survivants")

genre_regles.csv : 15 prédictions changées par les règles, 36.6% de survivants
gbc_regles.csv : 2 prédictions changées par les règles, 36.1% de survivants
